## 1. Preparação

Importação das bibliotecas e configuração do ambiente.

Utilizamos apenas bibliotecas de **leitura, tratamento de dados e visualização** —
nenhuma biblioteca que contenha o algoritmo KNN implementado é importada nesta etapa.
O `Scikit-learn` só será importado na **Célula 6**, exclusivamente para a comparação
com a implementação *hardcore*.

- `numpy` → estruturas de dados e operações matemáticas (distâncias, votação)
- `pandas` → leitura e tratamento do CSV
- `matplotlib` → visualização (matriz de confusão)
- `random` → embaralhamento dos dados no split (sem depender de bibliotecas de ML)

A *seed* é fixada para garantir a **reprodutibilidade** dos experimentos.

In [8]:
import numpy as np              
import pandas as pd             
import matplotlib.pyplot as plt 
import seaborn as sns           
import random                   

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (6, 5)

print("Bibliotecas importadas com sucesso.")

Bibliotecas importadas com sucesso.


## 2. Tratamento dos dados

Nesta etapa a base **Iris** é carregada a partir do arquivo CSV e separada em
conjuntos de **treino (70%)** e **teste (30%)**.

### Leitura e limpeza

O arquivo é lido com `pandas`. Caso exista a coluna `Id` (presente na versão do
Kaggle), ela é removida por ser apenas um identificador, sem valor preditivo.
Em seguida os dados são divididos em:

- `X` → a matriz de **atributos** (features): comprimento e largura da sépala e
  da pétala;
- `y` → o vetor de **classes** (a coluna `Species`).

Ambos são convertidos para arrays do `numpy`, o que facilita as operações
matemáticas nas etapas seguintes.

### Separação treino/teste (split 70/30)

O split é feito **manualmente**, sem utilizar bibliotecas de machine learning:

1. Cria-se uma lista com os índices de todas as amostras;
2. Essa lista é **embaralhada** aleatoriamente (usando a *seed* fixada na Célula 1,
   garantindo reprodutibilidade);
3. Os primeiros 30% dos índices formam o conjunto de **teste** e os 70% restantes
   o de **treino**.

O embaralhamento é importante porque a base Iris vem **ordenada por classe** — sem
ele, o conjunto de teste conteria apenas amostras das últimas classes, invalidando
a avaliação.

> **Configuração:** as variáveis `CAMINHO_CSV`, `COLUNA_CLASSE` e `PROP_TESTE`
> ficam no topo da célula para facilitar a reutilização do notebook com **outras
> bases de dados**.

In [9]:
CAMINHO_CSV = "Iris.csv"
COLUNA_CLASSE = "Species"
PROP_TESTE = 0.30

df = pd.read_csv(CAMINHO_CSV)

if "Id" in df.columns:
    df = df.drop(columns=["Id"])

X = df.drop(columns=[COLUNA_CLASSE]).to_numpy(dtype=float)
y = df[COLUNA_CLASSE].to_numpy()

indices = list(range(len(df)))
random.shuffle(indices)

n_teste = int(len(df) * PROP_TESTE)
indices_teste = indices[:n_teste]
indices_treino = indices[n_teste:]

X_treino, y_treino = X[indices_treino], y[indices_treino]
X_teste, y_teste = X[indices_teste], y[indices_teste]

print(f"Total de amostras: {len(df)}")
print(f"Treino: {len(X_treino)} amostras ({len(X_treino)/len(df):.0%})")
print(f"Teste:  {len(X_teste)} amostras ({len(X_teste)/len(df):.0%})")
print(f"Atributos (features): {X.shape[1]}")
print(f"Classes: {sorted(set(y))}")

Total de amostras: 150
Treino: 105 amostras (70%)
Teste:  45 amostras (30%)
Atributos (features): 4
Classes: ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
